## Objetivo

Demonstrar que toda imagem digital pode ser representada como uma **matriz numérica**, onde cada elemento contém os valores de cor (R, G, B) de um pixel.

## Metodologia

1. Carregar uma imagem e redimensioná-la para 50×50 pixels
2. Extrair cada pixel como uma tupla (Y, X, R, G, B)
3. Exportar esses dados para um arquivo CSV (planilha)
4. Reconstruir a imagem **exclusivamente** a partir do CSV
5. Comparar visualmente o resultado — provando que nenhuma informação foi perdida

Se a reconstrução for idêntica, fica provado que a imagem **é** a matriz de números.

## Passo 1 — Importação das Bibliotecas

Utilizamos:
- **Pillow (PIL)**: para manipulação de imagens (carregar, redimensionar, criar)
- **pandas**: para estruturar os dados dos pixels em formato tabular
- **NumPy**: para criar arrays numéricos eficientes na reconstrução
- **pathlib**: para manipulação de caminhos de arquivos
- **IPython.display**: para exibir as imagens diretamente no notebook

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

print("Bibliotecas importadas com sucesso.")

## Passo 2 — Definição dos Caminhos

Definimos os diretórios e caminhos de saída. A imagem de entrada deve estar na pasta `images/`.

In [ ]:
NOTEBOOK_DIR = Path(".").resolve()
IMAGES_DIR = NOTEBOOK_DIR / "images"
CSV_PATH = NOTEBOOK_DIR / "pixel_data.csv"
RECONSTRUCTED_PATH = NOTEBOOK_DIR / "reconstructed_image.png"

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

print(f"Diretório de imagens: {IMAGES_DIR}")
print(f"CSV de saída: {CSV_PATH}")
print(f"Imagem reconstruída: {RECONSTRUCTED_PATH}")

## Passo 3 — Carregar e Redimensionar a Imagem

Buscamos automaticamente a primeira imagem disponível na pasta `images/` e a redimensionamos para **50×50 pixels**.

O redimensionamento garante que a matriz resultante tenha um tamanho controlado (2.500 pixels = 2.500 linhas no CSV), facilitando a visualização e análise dos dados.

Também convertemos para o modo **RGB** (3 canais de cor), garantindo uniformidade independente do formato original.

In [ ]:
# Buscar a primeira imagem no diretório
image_path = None
for file in sorted(IMAGES_DIR.iterdir()):
    if file.suffix.lower() in SUPPORTED_EXTENSIONS:
        image_path = file
        break

if image_path is None:
    raise FileNotFoundError(
        "Nenhuma imagem encontrada na pasta 'images/'. "
        "Coloque um arquivo .png, .jpg, .jpeg ou .bmp lá e execute novamente."
    )

print(f"Imagem encontrada: {image_path.name}")

# Carregar e redimensionar
original = Image.open(image_path).convert("RGB")
print(f"Tamanho original: {original.size[0]}x{original.size[1]} pixels")

image = original.resize((50, 50))
print(f"Tamanho redimensionado: {image.size[0]}x{image.size[1]} pixels")

print("\nImagem redimensionada:")
display(image.resize((200, 200), Image.NEAREST))  # Ampliada para visualização

## Passo 4 — Extrair a Matriz de Pixels

Aqui está o núcleo da prova. Percorremos **cada pixel** da imagem, linha por linha (Y) e coluna por coluna (X), e extraímos seus valores de cor:

- **Y**: posição vertical (linha da matriz)
- **X**: posição horizontal (coluna da matriz)
- **R**: componente vermelho (0–255)
- **G**: componente verde (0–255)
- **B**: componente azul (0–255)

Cada pixel se torna uma linha no DataFrame. Uma imagem 50×50 gera exatamente **2.500 linhas**.

Isso demonstra que a imagem é, literalmente, uma tabela de números organizados em formato matricial.

In [ ]:
width, height = image.size
rows = []

for y in range(height):
    for x in range(width):
        r, g, b = image.getpixel((x, y))
        rows.append({"Y": y, "X": x, "R": r, "G": g, "B": b})

df = pd.DataFrame(rows)

print(f"Total de pixels extraídos: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print(f"\nPrimeiros 10 pixels (canto superior esquerdo):")
df.head(10)

### Estatísticas da Matriz

Podemos observar as estatísticas descritivas dos valores de cor, confirmando que todos estão no intervalo [0, 255].

In [ ]:
df[["R", "G", "B"]].describe()

## Passo 5 — Exportar para CSV (Planilha)

Salvamos o DataFrame como um arquivo CSV. Este arquivo contém **toda** a informação numérica da imagem.

O CSV pode ser aberto em qualquer editor de planilhas (Excel, Google Sheets, LibreOffice) para verificar que a imagem é, de fato, apenas uma tabela de números.

In [ ]:
df.to_csv(CSV_PATH, index=False)
print(f"Dados exportados para: {CSV_PATH.name}")
print(f"Tamanho do arquivo: {CSV_PATH.stat().st_size:,} bytes")
print(f"Linhas (pixels): {len(df)}")
print(f"Colunas: {list(df.columns)}")

## Passo 6 — Reconstruir a Imagem a Partir do CSV

Agora, **ignoramos completamente a imagem original**. Lemos apenas o arquivo CSV e reconstruímos a imagem pixel por pixel, usando somente os valores numéricos da tabela.

O processo:
1. Ler o CSV e determinar as dimensões da imagem (max(Y)+1 × max(X)+1)
2. Criar uma imagem em branco com essas dimensões
3. Para cada linha do CSV, definir o pixel na posição (X, Y) com a cor (R, G, B)

Se a imagem resultante for **idêntica** à original redimensionada, fica provado que **a matriz numérica contém 100% da informação da imagem**.

In [ ]:
# Ler o CSV (ignorando a imagem original)
df_csv = pd.read_csv(CSV_PATH).astype(int)

# Determinar dimensões
csv_height = df_csv["Y"].max() + 1
csv_width = df_csv["X"].max() + 1
print(f"Dimensões detectadas no CSV: {csv_width}x{csv_height} pixels")

# Criar array de pixels e preencher
pixels = np.zeros((csv_height, csv_width, 3), dtype=np.uint8)
for _, row in df_csv.iterrows():
    pixels[row["Y"], row["X"]] = [row["R"], row["G"], row["B"]]

# Criar imagem reconstruída
reconstructed = Image.fromarray(pixels)

# Salvar
reconstructed.save(RECONSTRUCTED_PATH)
print(f"Imagem reconstruída salva em: {RECONSTRUCTED_PATH.name}")

print("\nImagem reconstruída:")
display(reconstructed.resize((200, 200), Image.NEAREST))  # Ampliada para visualização

## Passo 7 — Comparação Visual

Exibimos lado a lado a imagem original (redimensionada) e a imagem reconstruída a partir do CSV para confirmar visualmente que são idênticas.

In [ ]:
# Comparação lado a lado
scale = 200
original_scaled = image.resize((scale, scale), Image.NEAREST)
reconstructed_scaled = reconstructed.resize((scale, scale), Image.NEAREST)

# Criar imagem de comparação
comparison = Image.new("RGB", (scale * 2 + 20, scale + 40), (255, 255, 255))
comparison.paste(original_scaled, (0, 40))
comparison.paste(reconstructed_scaled, (scale + 20, 40))

print("Original (esquerda) vs. Reconstruída do CSV (direita):")
display(comparison)

# Verificação numérica
original_array = np.array(image)
reconstructed_array = np.array(reconstructed)
are_identical = np.array_equal(original_array, reconstructed_array)

print(f"\nAs imagens são numericamente idênticas? {'SIM' if are_identical else 'NÃO'}")
if are_identical:
    print("\nConclusão: A imagem é inteiramente representada por sua matriz numérica.")
    print("Nenhuma informação foi perdida na conversão para números e vice-versa.")

## Passo 8 — Transformações Lineares Aplicadas à Imagem

Agora vamos além da representação: aplicamos **transformações lineares** à imagem, demonstrando que operações geométricas (rotação, reflexão, cisalhamento) são, na verdade, **multiplicações matriciais**.

Cada pixel da imagem pode ser tratado como um vetor de posição:

$$\mathbf{v} = \begin{bmatrix} x \\ y \end{bmatrix}$$

Uma transformação linear $T$ é representada por uma matriz $A$ tal que:

$$T(\mathbf{v}) = A \cdot \mathbf{v} = \begin{bmatrix} a & b \\ c & d \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix}$$

A seguir, aplicamos três transformações clássicas e analisamos seu efeito sobre a imagem.

### 8.1 — Reflexão Horizontal

A reflexão em torno do eixo vertical é dada pela matriz:

$$A_{\text{reflexão}} = \begin{bmatrix} -1 & 0 \\ 0 & 1 \end{bmatrix}$$

**Efeito geométrico:**
- O vetor $\mathbf{e}_1 = (1, 0)$ é mapeado para $(-1, 0)$ — inverte a direção horizontal
- O vetor $\mathbf{e}_2 = (0, 1)$ permanece inalterado
- Retas verticais ($x = c$) são mapeadas para $x = -c$
- A região da imagem é espelhada em torno do eixo $y$

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def apply_transform(img_array, matrix):
    """Aplica uma transformação linear 2x2 a cada posição de pixel da imagem."""
    h, w, c = img_array.shape
    # Centro da imagem (origem para a transformação)
    cx, cy = w / 2, h / 2
    result = np.zeros_like(img_array)
    # Matriz inversa para mapeamento reverso (de destino para origem)
    inv_matrix = np.linalg.inv(matrix)
    for y_dst in range(h):
        for x_dst in range(w):
            # Coordenadas relativas ao centro
            v = np.array([x_dst - cx, y_dst - cy])
            # Mapear de volta para a posição original
            src = inv_matrix @ v
            x_src = int(round(src[0] + cx))
            y_src = int(round(src[1] + cy))
            if 0 <= x_src < w and 0 <= y_src < h:
                result[y_dst, x_dst] = img_array[y_src, x_src]
    return result

def show_transform_effect(matrix, title):
    """Mostra o efeito da transformação nos vetores base e em uma região quadrada."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Efeito nos vetores ---
    ax = axes[0]
    ax.set_title(f"{title} — Efeito nos Vetores Base", fontsize=11)
    e1 = np.array([1, 0])
    e2 = np.array([0, 1])
    te1 = matrix @ e1
    te2 = matrix @ e2

    ax.quiver(0, 0, e1[0], e1[1], angles='xy', scale_units='xy', scale=1,
              color='blue', label=f'e₁ = {e1.tolist()}', width=0.03)
    ax.quiver(0, 0, e2[0], e2[1], angles='xy', scale_units='xy', scale=1,
              color='green', label=f'e₂ = {e2.tolist()}', width=0.03)
    ax.quiver(0, 0, te1[0], te1[1], angles='xy', scale_units='xy', scale=1,
              color='red', linestyle='dashed', label=f'T(e₁) = [{te1[0]:.2f}, {te1[1]:.2f}]', width=0.03)
    ax.quiver(0, 0, te2[0], te2[1], angles='xy', scale_units='xy', scale=1,
              color='orange', linestyle='dashed', label=f'T(e₂) = [{te2[0]:.2f}, {te2[1]:.2f}]', width=0.03)

    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.legend(fontsize=9, loc='upper left')

    # --- Efeito na região ---
    ax2 = axes[1]
    ax2.set_title(f"{title} — Efeito na Região Unitária", fontsize=11)

    # Quadrado original: vértices (0,0), (1,0), (1,1), (0,1)
    square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]]).T
    transformed = matrix @ square

    ax2.fill(square[0], square[1], alpha=0.3, color='blue', label='Região original')
    ax2.plot(square[0], square[1], 'b-', linewidth=2)
    ax2.fill(transformed[0], transformed[1], alpha=0.3, color='red', label='Região transformada')
    ax2.plot(transformed[0], transformed[1], 'r--', linewidth=2)

    ax2.set_xlim(-2, 2)
    ax2.set_ylim(-2, 2)
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axvline(x=0, color='k', linewidth=0.5)
    ax2.legend(fontsize=9)

    det = np.linalg.det(matrix)
    ax2.text(0.02, 0.02, f'det(A) = {det:.2f}\nÁrea transformada = {abs(det):.2f}× original',
             transform=ax2.transAxes, fontsize=9, verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    plt.show()

# --- Reflexão Horizontal ---
M_reflection = np.array([[-1, 0],
                          [ 0, 1]], dtype=float)

print("Matriz de Reflexão Horizontal:")
print(M_reflection)
print(f"\ndet(A) = {np.linalg.det(M_reflection):.0f} (preserva área, inverte orientação)\n")

img_array = np.array(image)
reflected = apply_transform(img_array, M_reflection)

show_transform_effect(M_reflection, "Reflexão Horizontal")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_array)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(reflected)
axes[1].set_title("Reflexão Horizontal")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### 8.2 — Rotação de 90°

A rotação anti-horária por um ângulo $\theta$ é dada pela matriz:

$$A_{\text{rotação}}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Para $\theta = 90°$:

$$A_{\text{rot 90°}} = \begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$$

**Efeito geométrico:**
- O vetor $\mathbf{e}_1 = (1, 0)$ é mapeado para $(0, 1)$ — gira 90° anti-horário
- O vetor $\mathbf{e}_2 = (0, 1)$ é mapeado para $(-1, 0)$
- Retas horizontais se tornam verticais e vice-versa
- A região da imagem é rotacionada, preservando área ($|\det(A)| = 1$)

In [ ]:
# --- Rotação de 90° anti-horária ---
theta = np.pi / 2
M_rotation = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])

print("Matriz de Rotação 90° anti-horária:")
print(np.round(M_rotation, 2))
print(f"\ndet(A) = {np.linalg.det(M_rotation):.0f} (preserva área e orientação)\n")

rotated = apply_transform(img_array, M_rotation)

show_transform_effect(M_rotation, "Rotação 90°")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_array)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(rotated)
axes[1].set_title("Rotação 90° anti-horária")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### 8.3 — Cisalhamento (Shear)

O cisalhamento horizontal por fator $k$ é dado pela matriz:

$$A_{\text{cisalhamento}} = \begin{bmatrix} 1 & k \\ 0 & 1 \end{bmatrix}$$

Para $k = 0{,}3$:

$$A_{\text{shear}} = \begin{bmatrix} 1 & 0.3 \\ 0 & 1 \end{bmatrix}$$

**Efeito geométrico:**
- O vetor $\mathbf{e}_1 = (1, 0)$ permanece inalterado
- O vetor $\mathbf{e}_2 = (0, 1)$ é mapeado para $(0.3, 1)$ — deslocado horizontalmente
- Retas horizontais permanecem horizontais, mas são deslocadas proporcionalmente à sua altura
- A região quadrada se deforma em um paralelogramo
- $|\det(A)| = 1$: a área é preservada, mas a forma muda

In [ ]:
# --- Cisalhamento Horizontal ---
k = 0.3
M_shear = np.array([[1, k],
                     [0, 1]], dtype=float)

print("Matriz de Cisalhamento Horizontal (k=0.3):")
print(M_shear)
print(f"\ndet(A) = {np.linalg.det(M_shear):.0f} (preserva área, deforma a região)\n")

sheared = apply_transform(img_array, M_shear)

show_transform_effect(M_shear, "Cisalhamento (k=0.3)")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_array)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(sheared)
axes[1].set_title("Cisalhamento Horizontal")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Resumo das Transformações

| Transformação | Matriz | det(A) | Efeito em Retas | Efeito em Regiões |
|---|---|---|---|---|
| Reflexão Horizontal | $\begin{bmatrix} -1 & 0 \\ 0 & 1 \end{bmatrix}$ | -1 | Retas verticais invertem posição | Espelhamento, área preservada |
| Rotação 90° | $\begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$ | 1 | Horizontais ↔ Verticais | Giro rígido, área preservada |
| Cisalhamento | $\begin{bmatrix} 1 & 0.3 \\ 0 & 1 \end{bmatrix}$ | 1 | Horizontais fixas, deslocamento proporcional | Quadrado → Paralelogramo, área preservada |

O **determinante** da matriz indica a variação de área: $|\det(A)| = 1$ significa que a área da região é preservada. Quando $\det(A) < 0$, a orientação é invertida (espelhamento).

## Conclusão

Demonstramos que:

1. Uma imagem digital de dimensão **H × W** é equivalente a uma **matriz H × W**, onde cada elemento é um vetor de 3 componentes (R, G, B)
2. Essa matriz pode ser serializada como uma tabela de números (CSV) sem perda de informação
3. A imagem pode ser perfeitamente reconstruída a partir da tabela numérica
4. **Transformações geométricas** sobre a imagem (reflexão, rotação, cisalhamento) são realizadas por **multiplicação matricial**, evidenciando que a Álgebra Linear é a linguagem matemática subjacente à manipulação de dados visuais

Portanto, **uma imagem digital é, por definição, uma matriz de números**, e toda operação visual sobre ela é uma operação algébrica sobre essa matriz.